In [50]:
import json
import os
from typing import Dict, Any, List, Optional
import pandas as pd
from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go

default_sample = """\
{"timestamp": 1759790081.822003, "device": "phone_2", "action": "idle", "time": 0.01679604311979629}
{"timestamp": 1759790081.822016, "time": 0.0799571066350519, "action": "transmit_end", "id": "all_gather_reduce_from_model_parallel_region_phone_1_phone_2_0", "internal_id": 1363, "duration": 0.07533652252252253}
{"timestamp": 1759790081.822017, "device": "phone_2", "action": "running", "time": 0.0799571066350519}
{"timestamp": 1759790081.846875, "time": 0.10479856463331555, "action": "transmit_start", "id": "all_gather_reduce_from_model_parallel_region_phone_2_phone_1_0", "internal_id": 1364, "size": 8286208.0}
{"timestamp": 1759790081.846889, "device": "phone_2", "action": "idle", "time": 0.1048207316385425}
{"timestamp": 1759790081.84691, "time": 0.09211773163821374, "action": "transmit_end", "id": "all_gather_reduce_from_model_parallel_region_phone_2_phone_1_0", "internal_id": 1362, "duration": 0.07533652252252253}
{"timestamp": 1759790081.846913, "device": "phone_1", "action": "running", "time": 0.09211773163821374}
{"timestamp": 1759790081.89344, "time": 0.13863189764794254, "action": "transmit_start", "id": "all_gather_reduce_from_model_parallel_region_phone_1_phone_2_0", "internal_id": 1365, "size": 8286208.0}
{"timestamp": 1759790081.8934531, "device": "phone_1", "action": "idle", "time": 0.1386457726371783}
{"timestamp": 1759790081.893468, "device": "phone_2", "action": "running", "time": 0.1048207316385425}
"""


def parse_src_dst(name: str) -> tuple[Optional[str], Optional[str]]:
    if not name:
        return None, None
    parts = name.split("_")
    try:
        idx = None
        for i in range(len(parts) - 1, -1, -1):
            if parts[i].isdigit():
                idx = i
                break
        if idx is None:
            return None, None

        def pop_name_num(j):
            if j - 1 >= 0 and parts[j - 1].isdigit() and j - 2 >= 0:
                return f"{parts[j - 2]}_{parts[j - 1]}", j - 2
            return parts[j - 1], j - 1

        dst, j = pop_name_num(idx)
        src, _ = pop_name_num(j)
        return src, dst
    except Exception:
        return None, None


def load_jsonl_or_sample(path: str) -> List[Dict[str, Any]]:
    events = []
    if os.path.exists(path):
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    events.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    if not events:
        events = [json.loads(line) for line in default_sample.strip().splitlines()]
    return events


events = load_jsonl_or_sample("../profile_out/event_log.jsonl")


starts: Dict[int, Dict[str, Any]] = {}
ends: Dict[int, Dict[str, Any]] = {}
meta: Dict[int, Dict[str, Any]] = {}

for ev in events:
    if not isinstance(ev, dict):
        continue
    action = ev.get("action", "")
    if not action.startswith("transmit_"):
        continue

    iid = ev.get("internal_id")
    if iid is None:
        continue

    if iid not in meta:
        meta[iid] = {}
    for k in ("id", "size"):
        if k in ev and k not in meta[iid]:
            meta[iid][k] = ev[k]

    if action == "transmit_start":
        starts[iid] = ev
    elif action == "transmit_end":
        ends[iid] = ev

rows = []
for iid, m in meta.items():
    s = starts.get(iid)
    e = ends.get(iid)

    name = m.get("id", f"internal_{iid}")
    size = m.get("size")
    src, dst = parse_src_dst(name)

    start_t = None
    end_t = None
    duration = None

    if s and "time" in s:
        start_t = float(s["time"])
    if e and "time" in e:
        end_t = float(e["time"])

    if s and "duration" in s:
        duration = float(s["duration"])
    if e and "duration" in e:
        duration = float(e["duration"])

    if start_t is not None and end_t is None and duration is not None:
        end_t = start_t + duration
    if end_t is not None and start_t is None and duration is not None:
        start_t = end_t - duration

    if (start_t is None or end_t is None) and e and ("time" in e) and ("duration" in e):
        end_t = float(e["time"])
        start_t = end_t - float(e["duration"])

    if start_t is None or end_t is None:
        continue

    rows.append(
        {
            "internal_id": iid,
            "name": name,
            "start": start_t,
            "end": end_t,
            "duration_s": end_t - start_t,
            "size_bytes": float(size) if size is not None else None,
            "src": src,
            "dst": dst,
        }
    )

df = pd.DataFrame(rows).sort_values(by=["start", "end", "internal_id"]).reset_index(drop=True)
df["internal_id_str"] = df["internal_id"].astype(str)
df["start"] = pd.to_numeric(df["start"], errors="coerce")
df["end"] = pd.to_numeric(df["end"], errors="coerce")

if df.empty:
    print("No intervals could be built from transmit events (need transmit_end with duration, or start+end pairs).")
else:
    fig = go.Figure()

    for name, dfg in df.groupby("name"):
        fig.add_bar(
            orientation="h",
            y=dfg["internal_id_str"],
            x=dfg["duration_s"],      # bar width = duration
            base=dfg["start"],        # bar start position
            name=name,
            hovertext=dfg.apply(
                lambda r: f"{r['name']}<br>ID:{r['internal_id']}<br>"
                        f"start:{r['start']:.6f}s <br>"
                        f"end:{r['end']:.6f}s<br>"
                        f"dur:{r['duration_s']:.6f}s <br>"
                        f"[src:{r['src']}] - [dst:{r['dst']}]<br>"
                        f"size: {r['size_bytes'] / 1024:.2f} KB" if r['size_bytes'] is not None else "size: N/A",
                axis=1,
            ),
            hoverinfo="text",
        )

    fig.update_yaxes(autorange="reversed", title="Internal ID")
    fig.update_xaxes(title="Simulation Time (s)", type="linear", rangeslider_visible=False)
    fig.update_layout(barmode="overlay", hovermode="closest", legend_title_text="Role (id)")
    display(fig)

In [ ]:
import pandas as pd
import re

data = """
[rank0] RTT (ms): min=0.673, max=2.466, mean=0.890, std=0.372
[rank0] Size=1B | Bandwidth (MB/s): min=0.00, max=0.00, mean=0.00, std=0.00
[rank0] Size=1B | Transfer time (s): min=0.000725, max=0.000851, mean=0.000763, std=0.000038
[rank0] Size=2B | Bandwidth (MB/s): min=0.00, max=0.01, mean=0.00, std=0.00
[rank0] Size=2B | Transfer time (s): min=0.000761, max=0.001647, mean=0.001112, std=0.000294
[rank0] Size=4B | Bandwidth (MB/s): min=0.01, max=0.01, mean=0.01, std=0.00
[rank0] Size=4B | Transfer time (s): min=0.000622, max=0.000747, mean=0.000693, std=0.000033
[rank0] Size=8B | Bandwidth (MB/s): min=0.02, max=0.03, mean=0.02, std=0.00
[rank0] Size=8B | Transfer time (s): min=0.000549, max=0.000740, mean=0.000669, std=0.000069
[rank0] Size=16B | Bandwidth (MB/s): min=0.02, max=0.05, mean=0.03, std=0.01
[rank0] Size=16B | Transfer time (s): min=0.000593, max=0.001972, mean=0.001186, std=0.000517
[rank0] Size=32B | Bandwidth (MB/s): min=0.03, max=0.10, mean=0.06, std=0.03
[rank0] Size=32B | Transfer time (s): min=0.000599, max=0.002107, mean=0.001273, std=0.000619
[rank0] Size=64B | Bandwidth (MB/s): min=0.13, max=0.20, mean=0.18, std=0.03
[rank0] Size=64B | Transfer time (s): min=0.000607, max=0.000921, mean=0.000710, std=0.000121
[rank0] Size=128B | Bandwidth (MB/s): min=0.29, max=0.41, mean=0.35, std=0.04
[rank0] Size=128B | Transfer time (s): min=0.000597, max=0.000854, mean=0.000705, std=0.000090
[rank0] Size=256B | Bandwidth (MB/s): min=0.59, max=0.83, mean=0.69, std=0.10
[rank0] Size=256B | Transfer time (s): min=0.000590, max=0.000829, mean=0.000718, std=0.000101
[rank0] Size=512B | Bandwidth (MB/s): min=0.59, max=1.38, mean=0.99, std=0.27
[rank0] Size=512B | Transfer time (s): min=0.000706, max=0.001649, mean=0.001070, std=0.000326
[rank0] Size=1KB | Bandwidth (MB/s): min=1.72, max=2.38, mean=2.09, std=0.22
[rank0] Size=1KB | Transfer time (s): min=0.000821, max=0.001135, mean=0.000945, std=0.000105
[rank0] Size=2KB | Bandwidth (MB/s): min=3.84, max=4.27, mean=4.07, std=0.12
[rank0] Size=2KB | Transfer time (s): min=0.000914, max=0.001019, mean=0.000961, std=0.000029
[rank0] Size=4KB | Bandwidth (MB/s): min=6.20, max=8.84, mean=7.26, std=0.77
[rank0] Size=4KB | Transfer time (s): min=0.000884, max=0.001261, mean=0.001089, std=0.000114
[rank0] Size=8KB | Bandwidth (MB/s): min=6.35, max=18.39, mean=12.62, std=3.76
[rank0] Size=8KB | Transfer time (s): min=0.000850, max=0.002460, mean=0.001374, std=0.000479
[rank0] Size=16KB | Bandwidth (MB/s): min=25.66, max=29.90, mean=27.41, std=1.11
[rank0] Size=16KB | Transfer time (s): min=0.001045, max=0.001218, mean=0.001142, std=0.000045
[rank0] Size=32KB | Bandwidth (MB/s): min=22.65, max=43.52, mean=35.07, std=8.16
[rank0] Size=32KB | Transfer time (s): min=0.001436, max=0.002759, mean=0.001902, std=0.000521
[rank0] Size=64KB | Bandwidth (MB/s): min=37.23, max=57.16, mean=48.34, std=6.75
[rank0] Size=64KB | Transfer time (s): min=0.002187, max=0.003357, mean=0.002641, std=0.000398
[rank0] Size=128KB | Bandwidth (MB/s): min=58.57, max=72.99, mean=64.95, std=4.79
[rank0] Size=128KB | Transfer time (s): min=0.003425, max=0.004268, mean=0.003870, std=0.000288
[rank0] Size=256KB | Bandwidth (MB/s): min=59.39, max=89.15, mean=80.46, std=7.41
[rank0] Size=256KB | Transfer time (s): min=0.005609, max=0.008419, mean=0.006281, std=0.000731
[rank0] Size=512KB | Bandwidth (MB/s): min=69.27, max=85.05, mean=78.60, std=4.41
[rank0] Size=512KB | Transfer time (s): min=0.011758, max=0.014436, mean=0.012764, std=0.000745
[rank0] Size=1MB | Bandwidth (MB/s): min=78.53, max=81.51, mean=80.26, std=1.02
[rank0] Size=1MB | Transfer time (s): min=0.024537, max=0.025469, mean=0.024922, std=0.000320
[rank0] Size=2MB | Bandwidth (MB/s): min=82.34, max=84.25, mean=83.46, std=0.63
[rank0] Size=2MB | Transfer time (s): min=0.047477, max=0.048582, mean=0.047930, std=0.000363
[rank0] Size=4MB | Bandwidth (MB/s): min=85.19, max=86.89, mean=86.09, std=0.46
[rank0] Size=4MB | Transfer time (s): min=0.092070, max=0.093911, mean=0.092927, std=0.000501
[rank0] Size=8MB | Bandwidth (MB/s): min=86.56, max=87.71, mean=87.17, std=0.36
[rank0] Size=8MB | Transfer time (s): min=0.182421, max=0.184838, mean=0.183545, std=0.000762
[rank0] Size=16MB | Bandwidth (MB/s): min=87.77, max=89.60, mean=88.54, std=0.55
[rank0] Size=16MB | Transfer time (s): min=0.357127, max=0.364597, mean=0.361438, std=0.002240
[rank0] Size=32MB | Bandwidth (MB/s): min=87.88, max=90.73, mean=89.11, std=0.84
[rank0] Size=32MB | Transfer time (s): min=0.705403, max=0.728269, mean=0.718242, std=0.006739
[rank0] Size=100MB | Bandwidth (MB/s): min=86.75, max=89.67, mean=88.12, std=0.70
[rank0] Size=100MB | Transfer time (s): min=2.230395, max=2.305373, mean=2.269840, std=0.018107
"""

rows = []
pattern = re.compile(
    r"Size=(\d+\w+) \| (Bandwidth \(MB/s\)|Transfer time \(s\)): min=([\d.]+), max=([\d.]+), mean=([\d.]+), std=([\d.]+)"
)

# Skip the latency line
for line in data.splitlines()[1:]:
    m = pattern.search(line)
    if m:
        size, metric, minv, maxv, mean, std = m.groups()
        rows.append({
            "Size": size,
            "Metric": metric,
            "Min": float(minv),
            "Max": float(maxv),
            "Mean": float(mean),
            "Std": float(std),
        })

df = pd.DataFrame(rows)
print(df.head(10))

  Size             Metric       Min       Max      Mean       Std
0   1B   Bandwidth (MB/s)  0.000000  0.000000  0.000000  0.000000
1   1B  Transfer time (s)  0.000725  0.000851  0.000763  0.000038
2   2B   Bandwidth (MB/s)  0.000000  0.010000  0.000000  0.000000
3   2B  Transfer time (s)  0.000761  0.001647  0.001112  0.000294
4   4B   Bandwidth (MB/s)  0.010000  0.010000  0.010000  0.000000
5   4B  Transfer time (s)  0.000622  0.000747  0.000693  0.000033
6   8B   Bandwidth (MB/s)  0.020000  0.030000  0.020000  0.000000
7   8B  Transfer time (s)  0.000549  0.000740  0.000669  0.000069
8  16B   Bandwidth (MB/s)  0.020000  0.050000  0.030000  0.010000
9  16B  Transfer time (s)  0.000593  0.001972  0.001186  0.000517


In [2]:
def size_to_bytes(s: str) -> int:
    units = {"B": 1, "KB": 1024, "MB": 1024*1024}
    m = re.match(r"(\d+)([KMG]?B)", s)
    if not m:
        return None
    val, unit = m.groups()
    return int(val) * units[unit]

df["SizeBytes"] = df["Size"].apply(size_to_bytes)

In [15]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go

# Extract transfer time rows
df_time = df[df["Metric"]=="Transfer time (s)"].copy()
x = df_time["SizeBytes"].values
y = df_time["Mean"].values

# Define model: latency + size/bandwidth
def model(size, alpha, beta):
    return alpha + beta * size

# Fit curve
params, _ = curve_fit(model, x, y, p0=[1e-4, 1e-9])
alpha, beta = params
print(f"Latency (alpha): {alpha:.6f} s")
print(f"Effective 1/bandwidth (beta): {beta:.2e} s/byte")
print(f"Bandwidth ~ {1/beta/1e6:.2f} MB/s")

# Create interactive plot
fig = go.Figure()

# Scatter plot for measured data
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='markers',
    name='Measured',
    marker=dict(color='blue')
))

# Line plot for fitted curve
x_fit = np.logspace(np.log10(min(x)), np.log10(max(x)), 100)
y_fit = model(x_fit, *params)
fig.add_trace(go.Scatter(
    x=x_fit,
    y=y_fit,
    mode='lines',
    name='Fitted curve',
    line=dict(color='red')
))

# Update layout
fig.update_layout(
    title="Transfer Time vs Message Size",
    xaxis=dict(
        title="Message Size (bytes)",
        type="log"
    ),
    yaxis=dict(
        title="Transfer Time (s)",
        type="linear"
    ),
    hovermode="closest"
)

fig.show()

Latency (alpha): 0.000887 s
Effective 1/bandwidth (beta): 2.16e-08 s/byte
Bandwidth ~ 46.27 MB/s
